In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

#ımport models
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

#plot style for visualizations

#set background of the plot white and with a grid
sns.set(style="whitegrid")
#make all future plots 10x6
plt.rcParams['figure.figsize'] = (10, 6)

#load the dataset
try:
    df = pd.read_csv("water_risk_dataset.csv")
    print("Dataset loaded successfully.")
    
except FileNotFoundError:
    print("file not found")

Dataset loaded successfully.


# CENG 463 HW 1– Water Resource Risk Classification
**Start Date:**  
**Due Date:** Month Dayth, 2026

## Dataset Overview
The dataset utilized in this assignment originates from the World Resources Institute (WRI) – Aqueduct Water Risk Atlas. It provides country-level indicators describing key hydrological and environmental factors, which are listed in the table below. The objective is to classify each country into a Water Resource Risk Category (0-4) using these indicators. Students are also expected to create two derived features — Composite Water Stress Index (CWSI) and Seasonal–Flood Interaction (SFI).

| Feature | Description |
|---------|-------------|
| gid_0 | Country Code |
| bws_score | Baseline Water Stress |
| gtd_score | Groundwater Depletion |
| drr_score | Drought Risk |
| rfr_score | River Flood Risk |
| sev_score | Seasonal Variability |
| w_awr_def_tot_cat | Target: Water Risk Category (0-4) |


## 1. Feature Engineering (35 pts)
Students are expected to create two new features based on the existing indicators:

1. **Composite Water Stress Index (CWSI):**
   CWSI combines baseline water stress, groundwater depletion, and drought risk.
   Formula: CWSI = 0.5 × bws_score + 0.3 × gtd_score + 0.2 × drr_score

2. **Seasonal–Flood Interaction (SFI):**
   SFI represents interaction between seasonal variability and river flood risk.
   Formula: SFI = sev_score × rfr_score


In [ ]:
# TODO: Create CWSI and SFI features
#create CWSI feature
df['CWSI'] = (0.5 * df['bws_score'] +
              0.3 * df['gtd_score'] +
              0.2 * df['drr_score'])
#create SFI feature
df['SFI'] = (df['sev_score'] + df['rfr_score'])

#We are simply constructing 2 new features, using the old features. Composite Water Stress Index (CWSI) and
#Seasonal–Flood Interaction (SFI). How they are computed is given in the homework instructions and we have simply implemented it.

## 2. Model Training & Evaluation (40 pts)
Train five classification models: Random Forest, SVM, KNN, Gaussian Naive Bayes, Logistic Regression.

*Hint: Use scaled data for SVM, KNN, Logistic Regression.*

In [14]:
# TODO: Split data into features X and target y
# TODO: Train models and evaluate accuracy
# TODO: Prepare evaluation table

#gid_0 feature is country code and it is not contributing to any ML model
#including it is dangerous because our selected model can memorize a country code's target
#to some value and always remember it instead of learning patterns.
#exclude gid_0 feature which can led to overfitting and exclude target feature
X = df.drop(['gid_0', 'w_awr_def_tot_cat'], axis=1)

#send targets to y
y = df['w_awr_def_tot_cat']

#train test split X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#print training and testing set shapes for both X and y
#features gid_0 and w_awr_def_tot_cat excluded
#features CWSI and SFI added to data frame
#so there should be total of 7 features in a single data point
print(f"Training set shape: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Testing set shape: X_test={X_test.shape}, y_test={y_test.shape}")

#initializing Random Forest Model
model_rf = RandomForestClassifier(random_state=42)

#using pipeline for scaling the data and initializing models
#to prevent memory leaks

#scale data and initialize Support Vector Machine model
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42))
])

#scale data and initialize K Nearest Neighbour model
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

#initializing Gaussian Naive Bayes Model
model_gnb = GaussianNB()

#scale data and initialize Logistic Regression model
pipe_logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(random_state=42, max_iter=1000))
])

#dictionary of models
models = {
    'Random Forest': model_rf,
    'SVM': pipe_svm,
    'KNN': pipe_knn,
    'Gaussian Naive Bayes': model_gnb,
    'Logistic Regression': pipe_logreg
}

#initialize a dictionary to store model's base results
models_base_results = {}

print("\n Models Base Results :")

for name, model in models.items():
    print(f"Training {name} model")
    model.fit(X_train, y_train)#train the model
    y_predict = model.predict(X_test)#predict for test data
    accuracy = accuracy_score(y_test, y_predict)#compute accuracy of test data
    models_base_results[name] = accuracy#send accuracy to dictionary for further use
    print(f"accuracy for model is: {accuracy:.4f}\n")

#evaluation table
#send models_base_results to a data frame
base_results_df = pd.DataFrame.from_dict(models_base_results, orient='index',columns=['Base Accuracy'])
print("\n-----Evaluation Table-----")
#sort the results and print
print(base_results_df.sort_values(by='Base Accuracy', ascending=False))



Training set shape: X_train=(3644, 7), y_train=(3644,)
Testing set shape: X_test=(911, 7), y_test=(911,)

 Models Base Results :
Training Random Forest model
accuracy for model is: 0.9188

Training SVM model
accuracy for model is: 0.7673

Training KNN model
accuracy for model is: 0.8079

Training Gaussian Naive Bayes model
accuracy for model is: 0.6158

Training Logistic Regression model
accuracy for model is: 0.6773


-----Evaluation Table-----
                      Base Accuracy
Random Forest              0.918771
KNN                        0.807903
SVM                        0.767289
Logistic Regression        0.677278
Gaussian Naive Bayes       0.615807


## 3. Hyperparameter Optimization (20 pts)
Tune each model using GridSearchCV with 5-fold CV.
Compare baseline and tuned results and report improvements.
Identify the model with highest tuned performance.

Hint: Use accuracy as scoring metric. Add classification report.

In [ ]:
# TODO: Define parameter grids for each model
# TODO: Perform GridSearchCV and compare results

## 4. Feature Importance Analysis (5 pts)
Choose one model and analyze feature importance. Present most influential features in a table and bar chart.

In [ ]:
# TODO: Select your best model (e.g., Random Forest)
# TODO: Train the model on the training data if not already trained
# TODO: Calculate feature importances
# TODO: Create a table of features sorted by importance
# TODO: Plot a bar chart of feature importances
# TODO: Optional: Comment on top 3-5 most influential features